# 01 · EDA — Captions & Justifications

Exploratory analysis of the textual outputs (captions and justifications) produced by persona-driven MLLM annotators.

**Goals**
- Understand text length distributions per condition, persona, and sentiment label
- Vocabulary richness and lexical diversity
- Label (perception tag) frequency and co-occurrence
- Baseline statistics to motivate subsequent analyses

In [1]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter

from src.config import FIGURES, OUTPUTS, FONT_SCALE, SENT_COLORS, SENT_COLORS_3
from src.data_loading import load_annotations, parse_demographics, create_profiles

MUTED = sns.color_palette("muted")
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams["figure.dpi"] = 150
FIGURES.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)


In [ ]:
df = load_annotations()
df = parse_demographics(df)
df = create_profiles(df)

print(f"Records : {len(df):,}")
print(f"Images  : {df['image_id'].nunique():,}")
print(f"Personas: {df['persona_id'].nunique():,}")
df.head(2)


## 0 · Justification structure: terms or sentences?

Before running topic modeling on justifications it is important to confirm they are full prose sentences, not keyword lists. The median length (18 words) already suggests this; the examples below confirm it.

In [ ]:
import textwrap

print(f"Median justification length : {df['justification_len'].median():.0f} words")
print(f"Median caption length       : {df['caption_len'].median():.0f} words")
print()

print("─" * 80)
print("Sample justifications (one per predicted-sentiment class)")
print("─" * 80)
for label in ["Negative", "SlightlyNegative", "Neutral", "SlightlyPositive", "Positive"]:
    sample = df[df["predicted_sentiment"] == label]["justification"].dropna().iloc[0]
    print(f"[{label:18s}] {textwrap.fill(sample, width=70, subsequent_indent=' ' * 22)}")
print()
print("→ Justifications are full sentences written in the persona's voice,")
print("  making sentence-embedding and topic modeling appropriate.")


## 1 · Sentiment label distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = df["predicted_sentiment"].value_counts().index.tolist()
bar_colors = [SENT_COLORS.get(l, MUTED[0]) for l in order]
sns.countplot(data=df, x="predicted_sentiment", order=order, palette=bar_colors, ax=ax)
ax.set_xlabel("Sentiment label")
ax.set_ylabel("Count")
# ax.set_title("Predicted sentiment distribution (baseline)")
ax.tick_params(axis="x", rotation=30)
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center", va="bottom", fontsize=10,
        xytext=(0, 3), textcoords="offset points",
    )
plt.tight_layout()
fig.savefig(FIGURES / "fig_eda_sentiment_dist.pdf")
fig.savefig(FIGURES / "fig_eda_sentiment_dist.png")
plt.show()


## 2 · Text length distributions (caption vs. justification)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(axes,
                           ["caption_len", "justification_len"],
                           ["Caption length (words)", "Justification length (words)"]):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color=MUTED[0])
    ax.set_xlabel(title)
    ax.set_ylabel("Count")
#     ax.set_title(f"Distribution of {title.split()[0].lower()} length")
    ax.axvline(df[col].median(), color="#d73027", linestyle="--",
               label=f"Median = {df[col].median():.0f}")
    ax.legend()

plt.tight_layout()
fig.savefig(FIGURES / "fig_eda_text_length_dist.pdf")
fig.savefig(FIGURES / "fig_eda_text_length_dist.png")
plt.show()


## 3 · Length by sentiment label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sentiment_order = ["Positive", "Neutral", "Negative"]

for ax, col, title in zip(axes,
                           ["caption_len", "justification_len"],
                           ["Caption length", "Justification length"]):
    bp_palette = [SENT_COLORS_3[s] for s in sentiment_order]
    sns.boxplot(data=df[df["predicted_sentiment"].isin(sentiment_order)],
                x="predicted_sentiment", y=col, order=sentiment_order,
                palette=bp_palette, ax=ax)
    ax.set_xlabel("Sentiment label")
    ax.set_ylabel("Words")
#     ax.set_title(f"{title} by sentiment")
    ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
fig.savefig(FIGURES / "fig_eda_length_by_sentiment.pdf")
fig.savefig(FIGURES / "fig_eda_length_by_sentiment.png")
plt.show()


## 4 · Top perception tags

In [ ]:
all_tags = [tag for tags in df["predicted_perceptions"] for tag in tags]
tag_counts = Counter(all_tags)
top_n = 20
top_tags = pd.DataFrame(tag_counts.most_common(top_n), columns=["tag", "count"])

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=top_tags, y="tag", x="count", color=MUTED[0], ax=ax)
ax.set_xlabel("Frequency")
ax.set_ylabel("Perception tag")
# ax.set_title(f"Top {top_n} perception tags")
plt.tight_layout()
fig.savefig(FIGURES / "fig_eda_top_perception_tags.pdf")
fig.savefig(FIGURES / "fig_eda_top_perception_tags.png")
plt.show()

top_tags.to_csv(OUTPUTS / "top_perception_tags.csv", index=False)
print(top_tags.to_string(index=False))


## 5 · Perception tag co-occurrence heatmap (top 15)

In [ ]:
top15 = [t for t, _ in tag_counts.most_common(15)]
co_matrix = pd.DataFrame(0, index=top15, columns=top15)

for tags in df["predicted_perceptions"]:
    present = [t for t in tags if t in top15]
    for i, a in enumerate(present):
        for b in present[i:]:
            co_matrix.loc[a, b] += 1
            if a != b:
                co_matrix.loc[b, a] += 1

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(co_matrix, dtype=bool))
sns.heatmap(
    co_matrix, mask=mask, cmap="YlOrRd", annot=True, fmt="d",
    linewidths=0.4, ax=ax, cbar_kws={"label": "Co-occurrence count"},
    annot_kws={"size": 8},
)
# ax.set_title("Perception tag co-occurrence (top 15 tags)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)
plt.tight_layout()
fig.savefig(FIGURES / "fig_eda_tag_cooccurrence.pdf")
fig.savefig(FIGURES / "fig_eda_tag_cooccurrence.png")
plt.show()


## 6 · Save EDA summary stats

In [ ]:
summary = df[["caption_len", "justification_len", "n_perceptions"]].describe().round(2)
print(summary)
summary.to_csv(OUTPUTS / "eda_summary_stats.csv")